In [1]:
# ==========================================
# 1. ADIM: UNSLOTH VE GEREKSİNİMLERİN KURULUMU
# ==========================================
# Unsloth, Llama 3.1 eğitimini %200 daha hızlı ve %70 daha az bellekle yapmamızı sağlar.

%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

print("✅ Kurulumlar Başarıyla Tamamlandı!")

In [ ]:
from unsloth import FastLanguageModel
import torch

# GPU belleğini yormamak için maksimum dizi uzunluğunu 2048 yapıyoruz (bizim cümlelerimiz için fazlasıyla yeterli)
max_seq_length = 2048
dtype = None # Otomatik algılama
load_in_4bit = True # Bellek tasarrufu için 4-bit kuantizasyon (Çok kritik!)

print("⏳ Llama 3.1 modeli indiriliyor, bu işlem 1-2 dakika sürebilir...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("✅ Llama 3.1 (8B) 4-bit modeli başarıyla yüklendi!")

In [ ]:
from datasets import load_dataset

# 1. Veriyi yükle
dataset = load_dataset("json", data_files="train.json", split="train")
print(f"📥 Veri seti yüklendi! Toplam satır: {len(dataset)}")

# Modele vereceğimiz değişmez sistem komutu
system_instruction = (
    "You are an expert text normalization and error correction AI. "
    "Correct any spelling, grammar, punctuation, and specific entity terminology errors in the text. "
    "Apply standard entity normalization rules (e.g., 'Turkey' to 'Türkiye'). "
    "If the input is already correct, return it exactly as is."
)

# 2. Llama 3.1'in anlayacağı "Alpaca Prompt" formatını tanımla
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Cümlenin bittiğini modele söylemek için şarttır

def formatting_prompts_func(examples):
    inputs       = examples["input"]   # JSON'daki 'input' anahtarı
    outputs      = examples["target"]  # JSON'daki 'target' anahtarını okuyoruz
    texts = []

    for input_text, output_text in zip(inputs, outputs):
        # system_instruction'ı buraya otomatik ekliyoruz
        text = alpaca_prompt.format(system_instruction, input_text, output_text) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# 3. Formatı tüm veri setine uygula
dataset = dataset.map(formatting_prompts_func, batched = True)
print("✨ Veri seti Llama 3.1 formatına başarıyla çevrildi!")
print("\n🔍 Örnek Bir Çıktı Görünümü:")
print(dataset[0]["text"])

In [ ]:
# ==========================================
# 4. ADIM: LORA AYARLARI VE EĞİTİM MOTORU
# ==========================================
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import torch

# 1. Validation (Doğrulama) Setini Yükle ve Formatla
val_dataset = load_dataset("json", data_files="validation.json", split="train")
val_dataset = val_dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Validation seti yüklendi! Toplam satır: {len(val_dataset)}")

# 2. LoRA Adaptörlerini Modele Enjekte Et (Sadece %1-2'lik kısmı eğiteceğiz)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Öğrenme kapasitesi (16 önerilir)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth 0 olmasını önerir
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Bellek tasarrufu
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 3. Eğitim Motorunu (Trainer) Kur
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,       # 7006 satırlık ana eğitim setimiz
    eval_dataset = val_dataset,    # 1501 satırlık doğrulama setimiz
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1, # 7000 veri için 1 tur (epoch) başlangıçta çok idealdir
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 50,
        eval_strategy = "steps",
        eval_steps = 100,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 4. FİŞE TAKIYORUZ: EĞİTİMİ BAŞLAT!
print("\n🚀 EĞİTİM BAŞLIYOR! (Bu işlem verinin büyüklüğüne göre 15-30 dk sürebilir)")
trainer_stats = trainer.train()

Unsloth: Already have LoRA adapters! We shall skip this step.
✅ Validation seti yüklendi! Toplam satır: 1501
Unsloth: Tokenizing ["text"] (num_proc=6): 100%
 7006/7006 [00:09<00:00, 2218.12 examples/s]
Unsloth: Tokenizing ["text"] (num_proc=6): 100%
 1501/1501 [00:05<00:00, 450.12 examples/s]

🚀 EĞİTİM BAŞLIYOR! (Bu işlem verinin büyüklüğüne göre 15-30 dk sürebilir)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,006 | Num Epochs = 1 | Total steps = 876
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
 [876/876 1:56:14, Epoch 1/1]
Step	Training Loss	Validation Loss
100	0.500266	0.490110
200	0.498972	0.481460
300	0.509019	0.477881
400	0.490939	0.474099
500	0.487852	0.471477
600	0.475472	0.469959
700	0.473214	0.468441
800	0.485477	0.467217

In [ ]:
# BU HÜCREYİ EĞİTİM TAMAMEN BİTTİĞİNDE ÇALIŞTIR!
model.save_pretrained_merged("llama3_entity_normalization_model", tokenizer, save_method = "merged_16bit")
print("✅ Model başarıyla 'llama3_entity_normalization_model' klasörüne kaydedildi!")

config.json: 100%
 947/947 [00:00<00:00, 79.1kB/s]
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Download complete: 
 23.9k/0.00 [09:12<00:00, 167kB/s]
Fetching 1 files: 100%
 1/1 [00:00<00:00,  8.18it/s]
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.

Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]
model-00001-of-00004.safetensors: 100%
 4.98G/4.98G [01:50<00:00, 49.6MB/s]

Unsloth: Preparing safetensor model files:  25%|██▌       | 1/4 [01:51<05:33, 111.00s/it]
model-00002-of-00004.safetensors: 100%
 5.00G/5.00G [01:18<00:00, 79.1MB/s]

Unsloth: Preparing safetensor model files:  50%|█████     | 2/4 [03:09<03:04, 92.16s/it]
model-00003-of-00004.safetensors: 100%
 4.92G/4.92G [00:59<00:00, 94.8MB/s]

Unsloth: Preparing safetensor model files:  75%|███████▌  | 3/4 [04:10<01:17, 77.55s/it]
model-00004-of-00004.safetensors: 100%
 1.17G/1.17G [00:11<00:00, 174MB/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [04:21<00:00, 65.37s/it]
Note: tokenizer.model not found (this is OK for non-SentencePiece models)

Unsloth: Merging weights into 16bit:   0%|          | 0/4 [00:00<?, ?it/s]
Unsloth: Merging weights into 16bit:  25%|██▌       | 1/4 [01:36<04:49, 96.65s/it]
Unsloth: Merging weights into 16bit:  50%|█████     | 2/4 [03:03<03:01, 90.61s/it]
Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [04:35<01:31, 91.28s/it]
Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:50<00:00, 72.58s/it]
Unsloth: Merge process complete. Saved to `/content/llama3_entity_normalization_model`
✅ Model başarıyla 'llama3_entity_normalization_model' klasörüne kaydedildi!

In [ ]:
# ==========================================
# 5. ADIM: EĞİTİLEN MODELİ CANLI TEST ETME
# ==========================================
# Modeli "Eğitim" modundan "Çıkarım (Cevap verme)" moduna alıyoruz
FastLanguageModel.for_inference(model)

# Test etmek istediğimiz TUZAKLI cümlemiz
# Hem terminoloji (turkey, CZECH REPUBLIC) hem de küçük/büyük harf hataları var.
test_text = "The economy of turkey is growing, and they signed a deal with the CZECH REPUBLIC. Also Macedonia and BURMA are discussing new trade agreements"

# Eğitimde kullandığımız sistem komutu ve formatın aynısı
inputs = tokenizer(
[
    alpaca_prompt.format(
        system_instruction, # Orijinal kuralımız
        test_text,          # Bozuk cümlemiz
        "",                 # Cevabı model verecek, o yüzden burayı boş bırakıyoruz
    )
], return_tensors = "pt").to("cuda")

print("🤖 Llama Düşünüyor...\n" + "="*50)

# Modeli çalıştırıp metni ürettiriyoruz
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)

# Sadece modelin ürettiği "Response" (Yanıt) kısmını alıp ekrana yazdırıyoruz
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
response = decoded_output.split("### Response:\n")[-1]

print(f"❌ Orijinal    : {test_text}")
print(f"✅ Düzeltilmiş : {response}")

🤖 Llama Düşünüyor...
==================================================
❌ Orijinal    : The economy of turkey is growing, and they signed a deal with the CZECH REPUBLIC. Also Macedonia and BURMA are discussing new trade agreements
✅ Düzeltilmiş : The economy of Türkiye is growing, and they signed a deal with the Czech Republic. Also Macedonia and Burma are discussing new trade agreements.

In [ ]:
import json
import random

# ==========================================
# 7. ADIM: TEST SETİ İLE KAPSAMLI DEĞERLENDİRME
# ==========================================

# Test verisini okuyoruz
with open("test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

# Farklı hata tiplerini görmek için rastgele 5 örnek seçelim
# (İstersen buradaki 5 sayısını 10 yapabilirsin)
samples = random.sample(test_data, 5)

FastLanguageModel.for_inference(model)

print("🧪 TEST VERİSİ İLE DEĞERLENDİRME BAŞLIYOR 🧪\n" + "="*50)

for i, sample in enumerate(samples, 1):
    input_text = sample["input"]
    target_text = sample["target"]
    error_type = sample["error_type"]

    # Llama'ya soruyu hazırlıyoruz
    prompt = alpaca_prompt.format(
        system_instruction,
        input_text,
        "" # Cevabı Llama dolduracak
    )

    # Tensörlere çevirip GPU'ya yolluyoruz
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Çıkarım (Inference) işlemi
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

    # Çıktıyı temizleyip sadece yanıt kısmını alıyoruz
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    response = decoded_output.split("### Response:\n")[-1].strip()

    print(f"🔹 Örnek {i} (Hata Tipi: {error_type})")
    print(f"❌ Input   : {input_text}")
    print(f"🎯 Target  : {target_text}")
    print(f"🤖 Llama   : {response}")
    print("-" * 50)

🧪 TEST VERİSİ İLE DEĞERLENDİRME BAŞLIYOR 🧪
==================================================
🔹 Örnek 1 (Hata Tipi: none)
❌ Input   : Though natural language processing tasks are closely intertwined, they can be subdivided into categories for convenience.
🎯 Target  : Though natural language processing tasks are closely intertwined, they can be subdivided into categories for convenience.
🤖 Llama   : Though natural language processing tasks are closely intertwined, they can be subdivided into categories for convenience.
--------------------------------------------------
🔹 Örnek 2 (Hata Tipi: none)
❌ Input   : Ninth-century ecumenical councils applied this regulation to the laity.
🎯 Target  : Ninth-century ecumenical councils applied this regulation to the laity.
🤖 Llama   : Ninth-century ecumenical councils applied this regulation to the laity.
--------------------------------------------------
🔹 Örnek 3 (Hata Tipi: none)
❌ Input   : Together with the Tories, they were the conservatives in the late 18th century United Kingdom.
🎯 Target  : Together with the Tories, they were the conservatives in the late 18th century United Kingdom.
🤖 Llama   : Together with the Tories, they were the conservatives in the late 18th century United Kingdom.
--------------------------------------------------
🔹 Örnek 4 (Hata Tipi: punctuation)
❌ Input   : Mexican rule ended following the American Conquest of California, part of the larger Mexican-American War
🎯 Target  : Mexican rule ended following the American Conquest of California, part of the larger Mexican-American War.
🤖 Llama   : Mexican rule ended following the American Conquest of California, part of the larger Mexican-American War.
--------------------------------------------------
🔹 Örnek 5 (Hata Tipi: punctuation)
❌ Input   : Arrays within expressions were effectively treated as pointers
🎯 Target  : Arrays within expressions were effectively treated as pointers.
🤖 Llama   : Arrays within expressions were effectively treated as pointers.
--------------------------------------------------